In [ ]:
import sys
import os
sys.path.append(os.path.abspath(".."))

import pandas as pd
import numpy as np
import torch
from torch.utils.data import DataLoader
import torch.nn as nn
import torch.optim as optim

from src.preprocess import *
from src.embeddings import *
from src.data_loader import *
from src.model import *

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

In [ ]:
import os
print(os.getcwd())

In [ ]:
news_cols = ['news_id','category','subcategory','title',
             'abstract','url','title_entities','abstract_entities']

beh_cols = ['imp_id','user_id','time','history','impressions']

news_df = pd.read_csv('data/MINDsmall_train/news.tsv', sep='\t', names=news_cols)
behaviors_df = pd.read_csv('data/MINDsmall_train/behaviors.tsv', sep='\t', names=beh_cols)

word2idx = build_vocab(news_df)
news_dict = build_news_dict(news_df, word2idx)

glove_path = 'data/glove/glove.6B.300d.txt'
embedding_matrix = load_glove_embeddings(glove_path, word2idx)

samples = create_nrms_samples(behaviors_df, news_dict)

print("Number of samples:", len(samples))
print(samples[0]["history"].shape)
print(samples[0]["candidates"].shape)

In [ ]:
dataset = NRMSDataset(samples)

train_loader = DataLoader(
    dataset,
    batch_size=32,
    shuffle=True
)

batch = next(iter(train_loader))

print(batch["history"].shape)
print(batch["candidates"].shape)
print(batch["label"].shape)

In [ ]:
model = NRMSModel(embedding_matrix, num_heads=8, head_dim=16).to(device)

optimizer = optim.Adam(model.parameters(), lr=1e-4)
criterion = nn.CrossEntropyLoss()

print(model)

In [ ]:
EPOCHS = 3
loss_history = []

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0

    for batch in train_loader:
        history = batch["history"].to(device)
        candidates = batch["candidates"].to(device)

        optimizer.zero_grad()

        scores = model(history, candidates)

        # positive candidate is always index 0
        target = torch.zeros(scores.size(0), dtype=torch.long).to(device)

        loss = criterion(scores, target)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    loss_history.append(avg_loss)

    print(f"Epoch {epoch+1}/{EPOCHS}, Loss: {avg_loss:.4f}")